In [1]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

TF_AVAILABLE = False
print("TensorFlow disabled – using baseline predictions only.")

if os.getcwd().split(os.sep)[1] == "home":
    PLATFORM = "local"
else:
    PLATFORM = "kaggle"

if PLATFORM == "local":
    LOAD_DATA_FROM = "./input/hms-harmful-brain-activity-classification"
else:
    LOAD_DATA_FROM = "/kaggle/input/hms-harmful-brain-activity-classification"

if not os.path.isdir(LOAD_DATA_FROM):
    fallback_path = os.path.join(
        ".", "data", "hms-harmful-brain-activity-classification"
    )
    if os.path.isdir(fallback_path):
        LOAD_DATA_FROM = fallback_path
        print(
            f"Data directory not found at default location – falling back to {LOAD_DATA_FROM}"
        )
    else:
        raise FileNotFoundError(
            f"Cannot locate data directory. Tried {LOAD_DATA_FROM} and {fallback_path}"
        )

df = pd.read_csv(os.path.join(LOAD_DATA_FROM, "train.csv"))
TARGETS = df.columns[-6:]
print("Train shape:", df.shape)
print("Targets", list(TARGETS))




TensorFlow disabled – using baseline predictions only.
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


In [2]:
def compute_baseline_distributions(train_df, targets):
    """Return patient distribution, overall distribution, and patient sample counts."""
    patient_sum = train_df.groupby("patient_id")[targets].sum()
    patient_dist = patient_sum.div(patient_sum.sum(axis=1), axis=0)
    overall = train_df[targets].sum()
    overall = overall / overall.sum()
    patient_counts = train_df.groupby("patient_id").size()
    return patient_dist, overall, patient_counts




In [3]:
def _kl_div(p, q):
    """Kullback‑Leibler divergence for two probability vectors."""
    eps = 1e-12
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    return np.sum(p * np.log(p / q))




In [4]:
if __name__ == "__main__":
    # Full‑training distributions (used for final test predictions)
    patient_dist_full, overall_dist_full, patient_counts_full = (
        compute_baseline_distributions(df, TARGETS)
    )

    # Validation split – compute distributions only on the training portion to avoid leakage
    val_frac = 0.20
    val = df.sample(frac=val_frac, random_state=42)
    train_split = df.drop(val.index)

    patient_dist_val, overall_dist_val, patient_counts_val = (
        compute_baseline_distributions(train_split, TARGETS)
    )

    true_counts = val[TARGETS].values
    true_dist = true_counts / true_counts.sum(axis=1, keepdims=True)

    EPS = 1e-12
    candidate_alphas = np.array([0, 1, 5, 10, 20, 50, 100, 200])
    best_alpha = candidate_alphas[0]
    best_kl = np.inf

    for alpha in candidate_alphas:
        preds = []
        for _, row in val.iterrows():
            pid = row["patient_id"]
            if pid in patient_dist_val.index:
                patient_p = patient_dist_val.loc[pid].values
                cnt = patient_counts_val.loc[pid]
                w = cnt / (cnt + alpha) if alpha > 0 else 1.0
            else:
                patient_p = overall_dist_val.values
                w = 0.0
            blended = w * patient_p + (1 - w) * overall_dist_val.values
            blended = blended + EPS
            blended = blended / blended.sum()
            preds.append(blended)
        preds = np.stack(preds)
        kl_vals = np.array([_kl_div(p, q) for p, q in zip(true_dist, preds)])
        avg_kl = kl_vals.mean()
        if avg_kl < best_kl:
            best_kl = avg_kl
            best_alpha = alpha

    print(f"Chosen smoothing α = {best_alpha} with validation KL ≈ {best_kl:.5f}")

    test = pd.read_csv(os.path.join(LOAD_DATA_FROM, "test.csv"))
    print("Test shape", test.shape)

    preds_all = np.zeros((len(test), len(TARGETS)), dtype=np.float32)

    for idx, row in test.iterrows():
        pid = row["patient_id"]
        if pid in patient_dist_full.index:
            patient_p = patient_dist_full.loc[pid].values
            cnt = patient_counts_full.loc[pid]
            w = cnt / (cnt + best_alpha) if best_alpha > 0 else 1.0
        else:
            patient_p = overall_dist_full.values
            w = 0.0
        blended = w * patient_p + (1 - w) * overall_dist_full.values
        blended = blended + EPS
        blended = blended / blended.sum()
        preds_all[idx] = blended.astype(np.float32)

    sub = pd.DataFrame({"eeg_id": test["eeg_id"].values})
    sub[TARGETS] = preds_all
    sub.to_csv("submission.csv", index=False)
    print("Submission shape", sub.shape)
    print(sub.head())

Chosen smoothing α = 1 with validation KL ≈ 0.46568
Test shape (9850, 3)
Submission shape (9850, 7)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.002086  0.016720  0.563292   0.006563   0.184904   
1  2578018731      0.002086  0.016720  0.563292   0.006563   0.184904   
2  2578018731      0.002086  0.016720  0.563292   0.006563   0.184904   
3  2578018731      0.002086  0.016720  0.563292   0.006563   0.184904   
4  1706196508      0.114723  0.299432  0.006815   0.138579   0.020609   

   other_vote  
0    0.226434  
1    0.226434  
2    0.226434  
3    0.226434  
4    0.419841  
